# Sweep browser

Interactively browse the initial conditions `obstacle_init.ipynb` generated
into `sweeps/<family>/*.json`, instead of scrolling through saved preview
PNGs. Pick a family, then narrow down by whichever params actually vary in
that family's sweep (read straight from `sweeps/<family>/index.json` --
nothing here is hardcoded to a particular matrix shape, so this stays correct
however the sweep cells in `obstacle_init.ipynb` change).

Run `obstacle_init.ipynb` first if `./sweeps` is empty. The generation and
browsing notebooks share `utils.py`'s `renderCasePreview`, so what you see
here is rendered exactly the same way as the grid previews in that
notebook.

In [ ]:
%matplotlib widget
import warpSPHCore_config as swc
from typing import Any
swc.configure(precision="float32", dim=Any)

import warpSPHCore as sph
import warp as wp; wp.init()

import os
import torch
if torch.cuda.is_available():
    os.environ['TORCH_CUDA_ARCH_LIST'] = f'{torch.cuda.get_device_properties(0).major}.{torch.cuda.get_device_properties(0).minor}'

import matplotlib.pyplot as plt
from ipywidgets import widgets
from IPython.display import display

from warpSPH import *
from warpSPH.runner import CaseSpec
from utils import FAMILY_CASE, loadSweepIndex, renderCasePreview

## Index

Only families with a `sweeps/<family>/index.json` show up -- families you
haven't generated yet are simply absent from the dropdown rather than
erroring.

In [ ]:
AVAILABLE_FAMILIES = sorted(f for f in FAMILY_CASE if os.path.exists(f'sweeps/{f}/index.json'))
if not AVAILABLE_FAMILIES:
    raise RuntimeError('No sweeps found under ./sweeps -- run obstacle_init.ipynb first.')
print('available families:', AVAILABLE_FAMILIES)

## Browser

`(any)` leaves an axis unconstrained; the "Match #" slider pages through
whatever still matches once the axis dropdowns have narrowed it down (a fully
pinned-down combination leaves exactly one match, since each family's sweep
is the full cartesian product of its axes).

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

In [ ]:
ANY = '(any)'

family_dropdown = widgets.Dropdown(options=AVAILABLE_FAMILIES, description='Family:',
                                   style={'description_width': 'initial'})
axis_dropdowns = {}  # param name -> Dropdown, rebuilt whenever the family changes
axis_box = widgets.VBox([])
row_slider = widgets.IntSlider(description='Match #:', min=0, max=0, value=0,
                               continuous_update=False, style={'description_width': 'initial'})
info_label = widgets.HTML()

current = {'family': None, 'index': None}


def _filtered_rows():
    rows = current['index']['rows']
    for name, dd in axis_dropdowns.items():
        if dd.value != ANY:
            rows = [r for r in rows if str(r.get(name)) == dd.value]
    return rows


def _redraw(*_):
    rows = _filtered_rows()
    row_slider.max = max(0, len(rows) - 1)
    row_slider.value = min(row_slider.value, row_slider.max)
    if not rows:
        info_label.value = '<b>No sweep entry matches this combination.</b>'
        return
    row = rows[row_slider.value]
    case = FAMILY_CASE[current['family']]
    spec = CaseSpec.load(row['file'])
    ax.cla()
    renderCasePreview(case, spec, ax, markerSize=3, title=row['label'])
    fig.canvas.draw_idle()
    info_label.value = (f"<b>{row['label']}</b> &mdash; match {row_slider.value + 1}/{len(rows)} "
                        f"&mdash; <code>{row['file']}</code>")


def _rebuild_axes(*_):
    family = family_dropdown.value
    idx = loadSweepIndex(family)
    current['family'], current['index'] = family, idx

    axis_dropdowns.clear()
    dds = []
    for name in idx['axes']:
        values = sorted({str(r.get(name)) for r in idx['rows']})
        dd = widgets.Dropdown(options=[ANY] + values, value=ANY, description=name,
                              style={'description_width': 'initial'})
        dd.observe(_redraw, names='value')
        axis_dropdowns[name] = dd
        dds.append(dd)
    axis_box.children = dds
    row_slider.value = 0
    _redraw()


family_dropdown.observe(_rebuild_axes, names='value')
row_slider.observe(_redraw, names='value')

_rebuild_axes()

display(widgets.VBox([family_dropdown, axis_box, row_slider, info_label]))